In [71]:
import pandas as pd
import numpy as np
import re
import json
import nltk
import tensorflow as tf
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Input
from tensorflow.keras.models import Model
from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.utils import class_weight

nltk.download('stopwords')
stop_words = set(stopwords.words('indonesian'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\T460s\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [72]:
# ==========================================
# 1. LOAD DATASET ASLI
# ==========================================
print("Membaca berkas CSV...")
df_hoax = pd.read_csv('dataset/tbh_cleaned_v3.csv')
df_hoax['label'] = 1 

df_antara = pd.read_csv('dataset/antaranews_cleaned_v3.csv')
df_detik = pd.read_csv('dataset/detik_cleaned_v3.csv')
df_kompas = pd.read_csv('dataset/kompas_cleaned_v3.csv')

# Gabungkan data fakta terlebih dahulu
df_fakta_all = pd.concat([df_antara, df_detik, df_kompas], ignore_index=True)
df_fakta_all['label'] = 0

Membaca berkas CSV...


In [73]:
# ==========================================
# 2. OTOMATIS RENAME KOLOM MENJADI 'text_raw' (PENCEGAH KEYERROR)
# ==========================================
def dapatkan_kolom_teks(df):
    for col in df.columns:
        if col.lower() in ['text', 'teks', 'content', 'text_raw', 'isi_berita', 'judul']:
            return col
    for col in df.columns:
        if df[col].dtype == 'object' and col != 'label':
            return col
    return None

kolom_hoax = dapatkan_kolom_teks(df_hoax)
kolom_fakta = dapatkan_kolom_teks(df_fakta_all)

# Ubah nama kolom asli menjadi 'text_raw' SEBELUM dibersihkan
df_hoax = df_hoax.rename(columns={kolom_hoax: 'text_raw'})
df_fakta_all = df_fakta_all.rename(columns={kolom_fakta: 'text_raw'})

In [74]:
# ==========================================
# 3. BERSIHKAN DATA KOSONG & DUPLIKAT
# ==========================================
df_hoax = df_hoax.dropna(subset=['text_raw']).drop_duplicates(subset=['text_raw']).reset_index(drop=True)
df_fakta_all = df_fakta_all.dropna(subset=['text_raw']).drop_duplicates(subset=['text_raw']).reset_index(drop=True)

In [75]:
# ==========================================
# 4. PROSES DOWNSAMPLING (SEIMBANG 50:50)
# ==========================================
print(f"Jumlah data Hoaks siap pakai: {len(df_hoax)} baris.")
print(f"Jumlah data Fakta siap pakai sebelum dipotong: {len(df_fakta_all)} baris.")

# Potong data fakta secara acak agar sama rata dengan jumlah data hoaks
df_fakta_balanced = df_fakta_all.sample(n=len(df_hoax), random_state=42).reset_index(drop=True)

# Gabungkan secara final
df_all = pd.concat([df_hoax[['text_raw', 'label']], df_fakta_balanced[['text_raw', 'label']]], ignore_index=True)

# Acak urutan data agar tercampur sempurna
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

print("\n=== HASIL AKHIR DATASET UNTUK TRAINING ===")
print(f"Total dataset gabungan: {len(df_all)} baris.")
print(df_all['label'].value_counts())

Jumlah data Hoaks siap pakai: 11740 baris.
Jumlah data Fakta siap pakai sebelum dipotong: 11989 baris.

=== HASIL AKHIR DATASET UNTUK TRAINING ===
Total dataset gabungan: 23480 baris.
label
0    11740
1    11740
Name: count, dtype: int64


In [76]:
# ==========================================
# 3. FUNGSI PEMBERSIH TEKS (BAHASA INDONESIA)
# ==========================================
def clean_text(text):
    if not isinstance(text, str): 
        return ""
    text = text.lower()
    
    # PERBAIKAN BUG URL: Mengubah URL menjadi token kata 'tautanpalsu'
    text = re.sub(r'http\S+|www\S+', 'tautanpalsu', text) 
    
    text = re.sub(r'@\w+|#\w+', '', text)      # Hapus Mention & Hashtag
    text = re.sub(r'[^a-z\s]', '', text)       # Hapus Angka & Simbol
    text = re.sub(r'\s+', ' ', text).strip()   # Hapus Spasi Berlebih
    tokens = text.split()
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(tokens)

print("Memulai proses pembersihan teks (Preprocessing)...")

# BARIS INI YANG TADI HILANG / MENYEBABKAN KEYERROR:
df_all['text_clean'] = df_all['text_raw'].apply(clean_text)

print("✅ Pembersihan selesai! Kolom 'text_clean' berhasil dibuat.")

Memulai proses pembersihan teks (Preprocessing)...
✅ Pembersihan selesai! Kolom 'text_clean' berhasil dibuat.


In [77]:
# ==========================================
# 4. TOKENISASI & EKSPOR TOKENIZER BARU
# ==========================================
MAX_LENGTH = 100
VOCAB_SIZE = 20000  

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<unk>")

# Sekarang baris ini dijamin aman karena 'text_clean' sudah pasti ada di atasnya
tokenizer.fit_on_texts(df_all['text_clean'])

# Ekspor tokenizer ke bentuk JSON untuk dibaca FastAPI nanti
tokenizer_json = tokenizer.to_json()
with open('tokenizer.json', 'w', encoding='utf-8') as f:
    json.dump(json.loads(tokenizer_json), f, ensure_ascii=False, indent=4)

print("============== RESULT ==============")
print("✅ Tokenizer BARU multidomain berhasil diekspor menjadi 'tokenizer.json'!")

# Konversi teks ke sekuens angka siap pakai untuk training model
sequences = tokenizer.texts_to_sequences(df_all['text_clean'])
X_padded = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post', truncating='post')
y = df_all['label'].values

============== RESULT ==============
✅ Tokenizer BARU multidomain berhasil diekspor menjadi 'tokenizer.json'!


In [78]:
# ==========================================
# 5. DEFINISI CUSTOM ATTENTION LAYER
# ==========================================
@tf.keras.utils.register_keras_serializable()
class AttentionLayer(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attention_weight',
            shape=(input_shape[-1], 1),
            initializer='random_normal',
            trainable=True
        )
        self.b = self.add_weight(
            name='attention_bias',
            shape=(input_shape[1], 1),
            initializer='zeros',
            trainable=True
        )
        super(AttentionLayer, self).build(input_shape)

    def call(self, x):
        e = tf.keras.backend.tanh(tf.keras.backend.dot(x, self.W) + self.b)
        a = tf.keras.backend.softmax(e, axis=1)
        output = x * a
        return tf.keras.backend.sum(output, axis=1), a

In [79]:
# ==========================================
# 6. MEMBANGUN ARSITEKTUR MODEL ML
# ==========================================
print("Membangun struktur model Deep Learning...")

inputs = Input(shape=(MAX_LENGTH,))
embedding = Embedding(input_dim=VOCAB_SIZE, output_dim=128)(inputs)
lstm_out = Bidirectional(LSTM(64, return_sequences=True))(embedding) 

spatial_out, attention_weights = AttentionLayer()(lstm_out)
outputs = Dense(1, activation='sigmoid')(spatial_out)

model_train = Model(inputs=inputs, outputs=outputs)
model_train.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

model_train.summary()

Membangun struktur model Deep Learning...


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding_4 (Embedding)         │ (None, 100, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 100, 128)       │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ attention_layer_4               │ [(None, 128), (None,   │           228 │
│ (AttentionLayer)                │ 100, 1)]               │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,659,173 (10.14 MB)

 Trainable params: 2,659,173 (10.14 MB)

 Non-trainable params: 0 (0.00 B)

In [80]:
# ==========================================
# 7. PROSES TRAINING MODEL (DI SINI TEMPAT KODE BARUMU!)
# ==========================================
# 1. Hitung bobot proporsional otomatis agar kelas 0 (Fakta) dan kelas 1 (Hoaks) seimbang
weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y),
    y=y
)
class_weights_dict = dict(enumerate(weights))
print(f"Bobot penyeimbang kelas otomatis: {class_weights_dict}")

# 2. Jalankan proses training dengan menyisipkan parameter class_weight
print("\nMemulai proses training model yang sudah distabilkan...")
history = model_train.fit(
    X_padded, 
    y, 
    epochs=5, 
    batch_size=64, 
    validation_split=0.2,
    class_weight=class_weights_dict # <-- Disisipkan di sini agar hasil tidak jomplang
)

Bobot penyeimbang kelas otomatis: {0: np.float64(1.0), 1: np.float64(1.0)}

Memulai proses training model yang sudah distabilkan...
Epoch 1/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 67s 201ms/step - accuracy: 0.8962 - loss: 0.2338 - val_accuracy: 0.9796 - val_loss: 0.0621
Epoch 2/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 59s 201ms/step - accuracy: 0.9875 - loss: 0.0379 - val_accuracy: 0.9874 - val_loss: 0.0432
Epoch 3/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 80s 193ms/step - accuracy: 0.9968 - loss: 0.0123 - val_accuracy: 0.9830 - val_loss: 0.0557
Epoch 4/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 81s 189ms/step - accuracy: 0.9987 - loss: 0.0057 - val_accuracy: 0.9834 - val_loss: 0.0693
Epoch 5/5
294/294 ━━━━━━━━━━━━━━━━━━━━ 59s 201ms/step - accuracy: 0.9991 - loss: 0.0037 - val_accuracy: 0.9836 - val_loss: 0.0692


In [81]:
# ==========================================
# 8. EKSPOR MODEL MULTI-OUTPUT
# ==========================================
model_inference = Model(inputs=inputs, outputs=[outputs, attention_weights])
model_inference.save("hoax_detector_xai.keras")
print("\n✅ Model Berhasil Disimpan!")


✅ Model Berhasil Disimpan!
